In [1]:
import pandas as pd
from sqlalchemy import create_engine
from transformers import pipeline
from tqdm import tqdm

In [2]:
df_news = pd.read_csv('filtered_tech_news_2018_2020.csv')

In [3]:
headline_col = 'Article_title'
ticker_col = 'Stock_symbol'

In [4]:
df_clean = df_news[['Date_Clean', ticker_col, headline_col]].copy()
df_clean.rename(columns={'Date_Clean': 'Date', ticker_col: 'Ticker', headline_col: 'Headline'}, inplace=True)
df_clean['Date'] = pd.to_datetime(df_clean['Date']).dt.date

In [5]:
print(f"Number of headlines to analize: {len(df_clean)}")

Number of headlines to analize: 6175


In [6]:
nlp = pipeline("sentiment-analysis", model="ProsusAI/finbert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [7]:
def analyze_sentiment(text):
    try:
        result = nlp(str(text)[:512])[0] 
        return pd.Series([result['label'], round(result['score'], 4)])
    except:
        return pd.Series(['neutral', 0.0])

In [8]:
tqdm.pandas(desc="Processing AI")

df_clean[['Sentiment', 'Confidence']] = df_clean['Headline'].progress_apply(analyze_sentiment)

Processing AI: 100%|██████████| 6175/6175 [03:28<00:00, 29.59it/s]


In [9]:
engine = create_engine('sqlite:///market_data.db')

df_clean.to_sql('news_sentiment', con=engine, if_exists='replace', index=False)

6175